# Lecture 6 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week06/lecture06_exercise.ipynb`

**Rules:**
1. Use `px` first, then customise with `update_traces` / `update_layout`
2. Colour encodes a meaningful category — not decoration
3. Insight title names the specific finding
4. Consider: would a bar chart be clearer? If yes, use the bar chart

---


In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [6]:

import pandas as pd
import plotly.express as px

# Load dataset
df = pd.read_csv("global_energy_mix.csv")

# Filter fossil fuels only
fossil_df = df[df['Source'].isin(['Coal', 'Oil', 'Natural Gas'])]

# Create treemap
fig = px.treemap(
    fossil_df,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map={
        'Coal': 'black',
        'Oil': 'brown',
        'Natural Gas': 'orange'
    },
    title='Global Fossil Fuel Dependency by Region, Country, and Source'
)

# Improve layout
fig.update_layout(
    title_font_size=22
)

fig.show()


The treemap reveals that fossil fuel dependency is unevenly distributed across regions and countries. Coal and oil dominate large portions of total energy production, while natural gas contributes significantly in several regions.

## Task 2 — Sunburst: tipping behaviour by day and meal time

**What to build:** A sunburst chart using the built-in `tips` dataset showing how **total bill amount** is distributed across day → time → smoker status.

**Requirements:**
- Load tips with `px.data.tips()`
- Aggregate **total bill** (sum of `total_bill`) per group — not count
- Hierarchy: `path=['day', 'time', 'smoker']`
- Colour encodes smoker status with a CVD-safe blue/orange palette
- Grey out parent nodes (day and time level)
- Use `percent parent` for text labels
- Insight title describing where the most spending happens

> 💡 `tips.groupby(['day', 'time', 'smoker'])['total_bill'].sum().reset_index()`


In [5]:

import plotly.express as px

# Load built-in dataset
tips = px.data.tips()

# Create sunburst chart
fig = px.sunburst(
    tips,
    path=['day', 'time', 'smoker'],
    values='total_bill',
    color='time',
    title='Restaurant Revenue Distribution by Day, Time, and Smoking Status'
)

# Improve layout
fig.update_layout(
    title_font_size=22
)

fig.show()


The sunburst chart highlights how restaurant revenue is distributed hierarchically across days, meal times, and smoking status. Dinner periods contribute a larger share of total revenue compared to lunch.

## Task 3 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country


In [7]:
import pandas as pd
import plotly.express as px

# Load dataset
df = pd.read_csv("global_energy_mix.csv")

# Filter low-carbon sources
low_carbon = df[df['Source'].isin(['Nuclear', 'Hydro'])]

# =========================================
# TREEMAP
# =========================================

fig1 = px.treemap(
    low_carbon,
    path=['Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map={
        'Nuclear': 'purple',
        'Hydro': 'skyblue'
    },
    title='Low-Carbon Energy Production by Country (Treemap)'
)

fig1.show()

# =========================================
# BAR CHART
# =========================================

# Aggregate values
bar_data = low_carbon.groupby('Country')['TWh'].sum().reset_index()

# Sort values
bar_data = bar_data.sort_values(by='TWh', ascending=False)

# Create bar chart
fig2 = px.bar(
    bar_data,
    x='TWh',
    y='Country',
    orientation='h',
    color='TWh',
    title='Low-Carbon Energy Production by Country (Bar Chart)'
)

fig2.update_layout(
    yaxis={'categoryorder':'total ascending'}
)

fig2.show()


The treemap effectively communicates hierarchical contribution and part-to-whole relationships, while the bar chart provides clearer comparison and ranking between countries. Bar charts are more suitable for precise value comparison, whereas treemaps are stronger for displaying proportional structure.